In [1]:
# ─── CELL 1: Install dependencies ──────────────────────────

!pip install -q transformers sentencepiece sacremoses

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 15.5 MB/s eta 0:00:00a 0:00:01


In [2]:
# ─── CELL 2: Imports ───────────────────────────────────────

import os
import math
import random
import warnings
import pandas as pd
import torch
import threading
from transformers import MarianMTModel, MarianTokenizer
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

In [3]:
# ─── CELL 3: Device setup ──────────────────────────────────

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Device: cuda
GPU: Tesla T4


In [4]:
# ─── CELL 4: Configuration ─────────────────────────────────

# Model
MODEL_NAME = "Helsinki-NLP/opus-mt-en-es"

# Paths
INPUT_PATH = "/kaggle/input/datasets/danielantoniudumitru/job-description-resume/job_applicant_dataset.csv"
OUTPUT_PATH = "/kaggle/working/job_applicant_dataset_es.csv"
CKPT_PATH = "/kaggle/working/translation_checkpoint.csv"

# Columns to translate
TRANSLATE_COLS = ["Resume", "Job Description", "Job Roles"]

# Columns to preserve exactly as-is
KEEP_COLS = [
    "Job Applicant Name", "Age", "Gender",
    "Race", "Ethnicity", "Best Match"
]

# Translation settings
MAX_CHUNK_TOKENS = 450
BEAM_SIZE = 4
TRANSLATION_BATCH_SIZE = 64

In [5]:
# ─── CELL 5: Load model and tokenizer ──────────────────────

print(f"\nLoading translation model: {MODEL_NAME} …")
tokenizer = MarianTokenizer.from_pretrained(MODEL_NAME)

NUM_GPUS = torch.cuda.device_count()
print(f"GPUs available: {NUM_GPUS}")
for i in range(NUM_GPUS):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

models = []
for i in range(NUM_GPUS):
    device = torch.device(f"cuda:{i}")
    m = MarianMTModel.from_pretrained(MODEL_NAME).to(device)
    m.eval()
    models.append((m, device))
    print(f"  Model loaded on GPU {i}")

print("All models loaded successfully.")
print(f"Vocabulary size: {tokenizer.vocab_size:,}")


Loading translation model: Helsinki-NLP/opus-mt-en-es …


tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/826k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

  Model loaded on GPU 0


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


  Model loaded on GPU 1
All models loaded successfully.
Vocabulary size: 65,001


In [6]:
# ─── CELL 6: Translation helper functions ──────────────────

def count_tokens(text, tokenizer):
    return len(tokenizer.encode(str(text), truncation=False))


def split_into_chunks(text, tokenizer, max_tokens=MAX_CHUNK_TOKENS):
    if not text or str(text).strip() == "" or str(text) == "nan":
        return [""]

    text = str(text)
    lines = text.split("\n")

    chunks = []
    current = []
    current_tok = 0

    for line in lines:
        line_tok = count_tokens(line, tokenizer)

        if line_tok > max_tokens:
            if current:
                chunks.append("\n".join(current))
                current, current_tok = [], 0

            words = line.split()
            sub_chunk = []
            sub_tok = 0

            for word in words:
                wt = count_tokens(word, tokenizer)
                if sub_tok + wt > max_tokens and sub_chunk:
                    chunks.append(" ".join(sub_chunk))
                    sub_chunk, sub_tok = [], 0
                sub_chunk.append(word)
                sub_tok += wt

            if sub_chunk:
                chunks.append(" ".join(sub_chunk))

        elif current_tok + line_tok > max_tokens and current:
            chunks.append("\n".join(current))
            current = [line]
            current_tok = line_tok

        else:
            current.append(line)
            current_tok += line_tok

    if current:
        chunks.append("\n".join(current))

    return chunks if chunks else [""]


def translate_text_full(text, tokenizer, model):
    if not text or str(text).strip() == "" or str(text) == "nan":
        return ""

    chunks = split_into_chunks(str(text), tokenizer)

    translated_chunks = []
    for chunk in chunks:
        if not chunk.strip():
            translated_chunks.append("")
            continue

        enc = tokenizer(
            [chunk],
            return_tensors = "pt",
            padding = True,
            truncation = True,
            max_length = MAX_CHUNK_TOKENS + 62
        ).to(DEVICE)

        with torch.no_grad():
            out = model.generate(
                **enc,
                num_beams = BEAM_SIZE,
                max_length = MAX_CHUNK_TOKENS + 62,
                early_stopping = True
            )

        translated_chunks.append(
            tokenizer.batch_decode(out, skip_special_tokens=True)[0]
        )

    return "\n".join(translated_chunks)


def _translate_chunks_on_device(chunks, tokenizer, model, device,
                                  batch_size, beam_size, max_len,
                                  results_list, start_idx):
    for i in range(0, len(chunks), batch_size):
        batch = chunks[i : i + batch_size]
        batch_clean = [c if c.strip() else " " for c in batch]

        enc = tokenizer(
            batch_clean,
            return_tensors = "pt",
            padding = True,
            truncation = True,
            max_length = max_len
        ).to(device)

        with torch.no_grad():
            out = model.generate(
                **enc,
                num_beams = beam_size,
                max_length = max_len,
                early_stopping = True
            )

        decoded = tokenizer.batch_decode(out, skip_special_tokens=True)

        for j, (orig, dec) in enumerate(zip(batch, decoded)):
            results_list[start_idx + i + j] = "" if not orig.strip() else dec


def translate_column_full(series, tokenizer, models, desc="Translating",
                           batch_size=64):
    texts = series.tolist()

    flat_chunks = []
    doc_chunk_map = []

    for text in texts:
        chunks = split_into_chunks(str(text), tokenizer)
        start = len(flat_chunks)
        flat_chunks.extend(chunks)
        doc_chunk_map.append((start, len(flat_chunks)))

    total_chunks = len(flat_chunks)
    print(f"  [{desc}] {len(texts)} documents → "
          f"{total_chunks} chunks total "
          f"(avg {total_chunks / max(len(texts), 1):.1f} chunks/doc)")

    num_gpus = len(models)
    chunk_size = math.ceil(total_chunks / num_gpus)
    gpu_splits = []
    for i in range(num_gpus):
        lo = i * chunk_size
        hi = min(lo + chunk_size, total_chunks)
        gpu_splits.append((lo, hi))

    translated_flat = [None] * total_chunks

    threads = []
    for i, (m, device) in enumerate(models):
        lo, hi = gpu_splits[i]
        if lo >= total_chunks:
            break
        t = threading.Thread(
            target = _translate_chunks_on_device,
            args = (
                flat_chunks[lo:hi],
                tokenizer, m, device,
                batch_size,
                BEAM_SIZE,
                MAX_CHUNK_TOKENS + 62,
                translated_flat,
                lo
            ),
            name = f"GPU-{i}"
        )
        threads.append(t)

    print(f"  Launching {len(threads)} GPU threads …")
    for i, t in enumerate(threads):
        lo, hi = gpu_splits[i]
        print(f"    GPU {i}: chunks {lo}–{hi-1} "
              f"({hi - lo} chunks, "
              f"~{math.ceil((hi - lo) / batch_size)} batches)")

    for t in threads:
        t.start()

    with tqdm(total=total_chunks, desc=desc) as pbar:
        done = 0
        while any(t.is_alive() for t in threads):
            newly_done = sum(r is not None for r in translated_flat)
            pbar.update(newly_done - done)
            done = newly_done
            import time; time.sleep(0.5)
        pbar.update(sum(r is not None for r in translated_flat) - done)

    for t in threads:
        t.join()

    results = []
    for start, end in doc_chunk_map:
        results.append("\n".join(translated_flat[start:end]))

    return results

In [7]:
# ─── CELL 7: Load dataset ──────────────────────────────────

print(f"\nLoading dataset from: {INPUT_PATH}")
df = pd.read_csv(INPUT_PATH)
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nValue counts (Best Match):\n{df['Best Match'].value_counts()}")
print(f"\nUnique Job Roles: {df['Job Roles'].nunique()}")
print(f"Sample Job Roles: {df['Job Roles'].unique()[:5].tolist()}")

for col in TRANSLATE_COLS + KEEP_COLS:
    assert col in df.columns, f"Column '{col}' not found in dataset!"

print("\nAll expected columns verified.")


Loading dataset from: /kaggle/input/datasets/danielantoniudumitru/job-description-resume/job_applicant_dataset.csv
Shape: (10000, 9)
Columns: ['Job Applicant Name', 'Age', 'Gender', 'Race', 'Ethnicity', 'Resume', 'Job Roles', 'Job Description', 'Best Match']

Value counts (Best Match):
Best Match
0    5150
1    4850
Name: count, dtype: int64

Unique Job Roles: 51
Sample Job Roles: ['Fitness Coach', 'Physician', 'Financial Analyst', 'Supply Chain Manager', 'Database Administrator']

All expected columns verified.


In [8]:
# ─── CELL 8: Estimate translation time ─────────────────────

avg_chunks_resume = math.ceil(
    df["Resume"].str.len().mean() / (MAX_CHUNK_TOKENS * 4)
)
avg_chunks_jd = math.ceil(
    df["Job Description"].str.len().mean() / (MAX_CHUNK_TOKENS * 4)
)

total_chunks_est = len(df) * (avg_chunks_resume + avg_chunks_jd + 1)
secs_per_chunk = 0.8

print(f"\nEstimated translation time:")
print(f"  Avg chunks per Resume         : ~{avg_chunks_resume}")
print(f"  Avg chunks per Job Description: ~{avg_chunks_jd}")
print(f"  Total chunks estimated        : ~{total_chunks_est:,}")
print(f"  Estimated total time          : ~{total_chunks_est * secs_per_chunk / 3600:.1f} hours")
print(f"  (Checkpoint saves after every column in case of timeout)")


Estimated translation time:
  Avg chunks per Resume         : ~2
  Avg chunks per Job Description: ~2
  Total chunks estimated        : ~50,000
  Estimated total time          : ~11.1 hours
  (Checkpoint saves after every column in case of timeout)


In [9]:
# ─── CELL 9: Resume from checkpoint if available ───────────

if os.path.exists(CKPT_PATH):
    print(f"\nCheckpoint found at: {CKPT_PATH}")
    print("Resuming from last saved state …")

    df_done = pd.read_csv(CKPT_PATH)
    start_row = len(df_done)
    df_remaining = df.iloc[start_row:].reset_index(drop=True).copy()

    print(f"  Already translated : {start_row:,} rows")
    print(f"  Remaining          : {len(df_remaining):,} rows")
else:
    print("\nNo checkpoint found — starting from row 0.")

    df_done = pd.DataFrame(columns=df.columns)
    df_remaining = df.copy()
    start_row = 0


No checkpoint found — starting from row 0.


In [10]:
# ─── CELL 10: Translate ────────────────────────────────────

if len(df_remaining) == 0:
    print("\nAll rows already translated. Loading checkpoint as final output.")
    df_done = pd.read_csv(CKPT_PATH)
else:
    print(f"\nTranslating {len(df_remaining):,} remaining rows …")
    print("Checkpoint is saved after each column completes.\n")

    for col in TRANSLATE_COLS:
        print(f"\n{'─'*55}")
        print(f"  Column : {col}")
        print(f"{'─'*55}")

        translated = translate_column_full(
            df_remaining[col],
            tokenizer,
            models,
            desc=col,
            batch_size=64
        )
        df_remaining[col] = translated

        df_checkpoint = pd.concat(
            [df_done, df_remaining], ignore_index=True
        )
        df_checkpoint.to_csv(CKPT_PATH, index=False)
        print(f"  Column '{col}' done. "
              f"Checkpoint saved ({len(df_checkpoint):,} rows total).")

    df_done = pd.concat([df_done, df_remaining], ignore_index=True)


Translating 10,000 remaining rows …
Checkpoint is saved after each column completes.


───────────────────────────────────────────────────────
  Column : Resume
───────────────────────────────────────────────────────
  [Resume] 10000 documents → 19993 chunks total (avg 2.0 chunks/doc)
  Launching 2 GPU threads …
    GPU 0: chunks 0–9996 (9997 chunks, ~157 batches)
    GPU 1: chunks 9997–19992 (9996 chunks, ~157 batches)


Resume:   0%|          | 0/19993 [00:00<?, ?it/s]

  Column 'Resume' done. Checkpoint saved (10,000 rows total).

───────────────────────────────────────────────────────
  Column : Job Description
───────────────────────────────────────────────────────
  [Job Description] 10000 documents → 10000 chunks total (avg 1.0 chunks/doc)
  Launching 2 GPU threads …
    GPU 0: chunks 0–4999 (5000 chunks, ~79 batches)
    GPU 1: chunks 5000–9999 (5000 chunks, ~79 batches)


Job Description:   0%|          | 0/10000 [00:00<?, ?it/s]

  Column 'Job Description' done. Checkpoint saved (10,000 rows total).

───────────────────────────────────────────────────────
  Column : Job Roles
───────────────────────────────────────────────────────
  [Job Roles] 10000 documents → 10000 chunks total (avg 1.0 chunks/doc)
  Launching 2 GPU threads …
    GPU 0: chunks 0–4999 (5000 chunks, ~79 batches)
    GPU 1: chunks 5000–9999 (5000 chunks, ~79 batches)


Job Roles:   0%|          | 0/10000 [00:00<?, ?it/s]

  Column 'Job Roles' done. Checkpoint saved (10,000 rows total).


In [11]:
# ─── CELL 11: Save final output ────────────────────────────

df_done.to_csv(OUTPUT_PATH, index=False)

print(f"\n{'='*60}")
print(f"  Translation complete!")
print(f"  Output saved to : {OUTPUT_PATH}")
print(f"  Final shape     : {df_done.shape}")
print(f"{'='*60}")


  Translation complete!
  Output saved to : /kaggle/working/job_applicant_dataset_es.csv
  Final shape     : (10000, 9)


In [12]:
# ─── CELL 12: Sanity check ─────────────────────────────────

print("\n─── Sample translations (row 0) ───")

for col in TRANSLATE_COLS:
    print(f"\n[{col}]")
    print(df_done[col].iloc[0][:400])

print("\n─── Preserved columns (must be identical to original) ───")

df_orig = pd.read_csv(INPUT_PATH)

for col in KEEP_COLS:
    match = (df_done[col].astype(str) == df_orig[col].astype(str)).all()
    status = "✓ unchanged" if match else "✗ CHANGED — investigate!"
    print(f"  {col:<30}: {status}")

print("\n─── Label distribution (must match original) ───")
print("Original :")
print(df_orig["Best Match"].value_counts().to_string())
print("Translated:")
print(df_done["Best Match"].value_counts().to_string())

print("\n─── Job Roles: English → Spanish samples ───")

role_pairs = list(zip(
    df_orig["Job Roles"].unique()[:10],
    df_done["Job Roles"].unique()[:10]
))

for orig, trans in role_pairs:
    print(f"  {str(orig):<35} → {trans}")


─── Sample translations (row 0) ───

[Resume]
Adquirir información sobre la salud y la salud en el lugar de trabajo.Daisuke Mori 243 Hill Street Amsterdam, North Holland +31 20 018-1590 daisuke_mori@protonmail.com https://www.linkedin.com/in/daisukemori --- RESUMEN PROFESSIONAL Fitness Coach con 2-4 años de experiencia práctica en Prevención de lesiones, Motivación, Nutrición.Adecuado para aplicar conocimientos técnicos para resolver problema

[Job Description]
Fitness Coach A Fitness Coach es responsable de ayudar a los clientes a alcanzar sus objetivos de fitness diseñando y liderando programas de fitness individuales o grupales. Usted proporcionará instrucción sobre ejercicios, forma adecuada y técnicas de prevención de lesiones, animando a los clientes a empujar sus límites mientras mantienen un enfoque en su bienestar. Acerca del rol El papel requie

[Job Roles]
Entrenador de fitness

─── Preserved columns (must be identical to original) ───
  Job Applicant Name            : ✓ un